# Prepare existing SynthHuman data as training shards

Use a **CPU runtime** for this one-time preparation. The notebook reads your existing Drive dataset through the API; it does not acquire the original release again. It reuses the saved inventory and selects the same 50,000 complete sample groups with seed 42. Source files remain available.

Run the setup, authentication, inventory, and preprocessing cells in order. In the final preparation cell set RUN_PREPARATION=True. The default PREPARE_SHARD_LIMIT=2 packs 500 training samples as a preparation benchmark. Then set PREPARE_SHARD_LIMIT=None and rerun that cell to finish all 50,000 samples, skipping finished shards. Each archive contains 250 samples; the full run produces 140 training, 30 validation, and 30 test archives. A partial benchmark alone is not enough for full training.

Progress is saved after each completed shard. An interrupted shard is rebuilt. Samples are resized to 256×256, masks/geometry are validated, and geometric targets retain float32 precision. RGB is stored as uint8; ImageNet normalization and flips happen during training. Initial preparation can still be slow: measure the 500-sample run before estimating full preparation time.

In [ ]:
import importlib.util
missing=[name for name in ('torch','numpy','PIL','cv2') if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing dependencies: '+', '.join(missing)+'. Install them in a separate Colab cell before continuing.')
print('Preparation dependencies are available. Select Runtime > Change runtime type > CPU.')
if importlib.util.find_spec("OpenEXR") is None:
    raise RuntimeError('In a separate Colab cell run: %pip install "OpenEXR>=3.3,<4", then rerun this dependency check.')


In [ ]:
import os,json,math,random,time
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
SEED=42
INPUT_SIZE=(256,256)
DATASET_MODE='pilot'
PILOT_SAMPLE_COUNT=50_000
EXPECTED_SYNTHHUMAN_SAMPLES=298008
VAL_FRACTION=TEST_FRACTION=0.15
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Preparing',PILOT_SAMPLE_COUNT,'samples at',INPUT_SIZE,'on CPU; no model training in this notebook.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/AppleCapstoneRelighting')
DATA_ROOT=DRIVE_ROOT/'dataset'
MANIFEST=DATA_ROOT/'pilot_manifests'/f'synthhuman_available_{PILOT_SAMPLE_COUNT}_seed{SEED}_v3.jsonl'
OUTPUT_DIR=DRIVE_ROOT/'runs'/'shard_preparation'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print('Existing dataset and outputs:',DRIVE_ROOT)

### Authenticate and index existing Drive files

In [ ]:
# 2a. Connect to the existing dataset through the Drive API.
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from googleapiclient.errors import HttpError
import httplib2
import google_auth_httplib2

SOURCE_FOLDER_ID = ''  # Optional: ID from the URL of your existing SynthHuman folder.
auth.authenticate_user()
credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
DRIVE_API = build('drive', 'v3', http=google_auth_httplib2.AuthorizedHttp(credentials, http=httplib2.Http(timeout=120)), cache_discovery=False)

def drive_query_literal(value):
    return str(value).replace('\\', '\\\\').replace("'", "\\'")

def find_drive_child(parent_id, name):
    query = f"'{drive_query_literal(parent_id)}' in parents and name = '{drive_query_literal(name)}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    response = DRIVE_API.files().list(q=query, pageSize=100, fields='files(id,name),nextPageToken', spaces='drive').execute(num_retries=5)
    matches = response.get('files', [])
    if len(matches) != 1 or response.get('nextPageToken'):
        raise ValueError(f'Folder {name!r}: found {len(matches)} matches. Set SOURCE_FOLDER_ID to the actual SynthHuman folder ID.')
    return matches[0]['id']

if not SOURCE_FOLDER_ID:
    SOURCE_FOLDER_ID = 'root'
    for folder_name in ('AppleCapstoneRelighting', 'dataset', 'sources', 'SynthHuman'):
        SOURCE_FOLDER_ID = find_drive_child(SOURCE_FOLDER_ID, folder_name)
folder = DRIVE_API.files().get(fileId=SOURCE_FOLDER_ID, fields='id,name,mimeType,trashed').execute(num_retries=5)
if folder.get('trashed') or folder['mimeType'] != 'application/vnd.google-apps.folder':
    raise ValueError('SOURCE_FOLDER_ID must point to the existing, non-trashed dataset folder.')
SYNTHHUMAN_ROOT = DATA_ROOT / 'sources' / 'SynthHuman'
DATA_IO_MODE = 'drive_api'
print('Existing dataset folder:', folder['name'], '| reader:', DATA_IO_MODE)
print('Next: API filename indexing only. File content is fetched on demand during the loader audit/training.')

# Check the same RGB path that failed through FUSE before doing the large index pass.
import io
PROBE_SAMPLE_ID = '0000004'
probe_name = f'rgb_{PROBE_SAMPLE_ID}.png'
probe_query = f"'{drive_query_literal(SOURCE_FOLDER_ID)}' in parents and name = '{probe_name}' and trashed = false"
probe_matches = DRIVE_API.files().list(q=probe_query, fields='files(id,name),nextPageToken', pageSize=10).execute(num_retries=5)
if len(probe_matches.get('files', [])) != 1 or probe_matches.get('nextPageToken'):
    raise FileNotFoundError(f'API could not identify one unique {probe_name}. Check SOURCE_FOLDER_ID or choose a known PROBE_SAMPLE_ID before indexing.')
probe_start = time.perf_counter()
probe_bytes = DRIVE_API.files().get_media(fileId=probe_matches['files'][0]['id']).execute(num_retries=5)
with Image.open(io.BytesIO(probe_bytes)) as probe_image:
    probe_image.load()
    print('API RGB read passed:', probe_name, '| image size:', probe_image.size, '| seconds:', round(time.perf_counter()-probe_start, 2))
del probe_bytes

In [ ]:
# Replace the notebook cell labeled "3. Index..." with this cell.
# Requires the runtime settings, Drive paths, and Drive API setup cells first.
from tqdm.auto import tqdm
import re

if DATASET_MODE != 'pilot':
    raise ValueError('Keep DATASET_MODE="pilot" for this experiment.')
INDEX_PATH = DATA_ROOT / 'pilot_manifests' / 'synthhuman_inventory_v3.json'
INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
index_config = {'version': 3, 'folder_id': SOURCE_FOLDER_ID}
state = {'config': index_config, 'files': {}, 'folders': [SOURCE_FOLDER_ID],
         'folder_position': 0, 'page_token': None, 'enumeration_done': False,
         'scanned': 0, 'ambiguous_names': [], 'empty_file_ids': []}
if INDEX_PATH.is_file():
    saved = json.loads(INDEX_PATH.read_text())
    if saved.get('config') == index_config:
        state = saved
all_files = state['files']
seen_folders = set(state['folders'])
ambiguous_names = set(state['ambiguous_names'])
empty_file_ids = set(state['empty_file_ids'])
pattern = re.compile(r'^(rgb|alpha|depth|normal)_(\d+)\.(png|exr)$')
required = {'rgb', 'alpha', 'depth', 'normal'}

def save_inventory():
    state['ambiguous_names'] = sorted(ambiguous_names)
    state['empty_file_ids'] = sorted(empty_file_ids)
    temporary = INDEX_PATH.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(state))
    temporary.replace(INDEX_PATH)

if not state['enumeration_done']:
    pages = 0
    progress = tqdm(desc='Inventory existing Drive files + subfolders', unit='entry', initial=state['scanned'])
    try:
        while state['folder_position'] < len(state['folders']):
            folder_id = state['folders'][state['folder_position']]
            kwargs = {'q': f"'{drive_query_literal(folder_id)}' in parents and trashed = false",
                      'pageSize': 1000, 'spaces': 'drive',
                      'fields': 'nextPageToken,incompleteSearch,files(id,name,size,mimeType)'}
            if state['page_token']:
                kwargs['pageToken'] = state['page_token']
            try:
                response = DRIVE_API.files().list(**kwargs).execute(num_retries=5)
            except HttpError as error:
                if error.resp.status == 400 and state['page_token']:
                    state['page_token'] = None
                    save_inventory()
                    print('Expired page token: restarting this folder, keeping indexed file IDs.')
                    continue
                raise
            if response.get('incompleteSearch'):
                raise RuntimeError('Drive returned an incomplete search; retry indexing before selecting training data.')
            items = response.get('files', [])
            for item in items:
                if item.get('mimeType') == 'application/vnd.google-apps.folder':
                    if item['id'] not in seen_folders:
                        state['folders'].append(item['id'])
                        seen_folders.add(item['id'])
                    continue
                name = item['name']
                match = pattern.fullmatch(name)
                if not match:
                    continue
                kind, _, extension = match.groups()
                if extension != ('png' if kind in ('rgb', 'alpha') else 'exr'):
                    continue
                size = int(item.get('size', 0))
                if size <= 0:
                    empty_file_ids.add(item['id'])
                    continue
                existing = all_files.get(name)
                if existing and existing['id'] != item['id']:
                    ambiguous_names.add(name)
                    continue
                all_files[name] = {'id': item['id'], 'size': size}
            state['scanned'] += len(items)
            progress.update(len(items))
            state['page_token'] = response.get('nextPageToken')
            if not state['page_token']:
                state['folder_position'] += 1
            progress.set_postfix(files=len(all_files), folder=f"{state['folder_position']}/{len(state['folders'])}")
            pages += 1
            if pages % 100 == 0:
                save_inventory()
        state['enumeration_done'] = True
    finally:
        save_inventory()
        progress.close()

# Select replacements from complete, unambiguous sample groups actually found.
groups = {}
ambiguous_samples = set()
for name, metadata in all_files.items():
    kind, sample_id, _ = pattern.fullmatch(name).groups()
    sample_id = int(sample_id)
    groups.setdefault(sample_id, {})[kind] = name
    if name in ambiguous_names:
        ambiguous_samples.add(sample_id)
complete_ids = sorted(sample_id for sample_id, kinds in groups.items()
                      if required <= kinds.keys() and sample_id not in ambiguous_samples)
incomplete = [{'sample_id': sample_id, 'missing': sorted(required - kinds.keys())}
              for sample_id, kinds in groups.items() if not required <= kinds.keys()]
report = {'files_indexed': len(all_files), 'folders_found': len(state['folders']),
          'complete_samples': len(complete_ids), 'incomplete_samples': len(incomplete),
          'ambiguous_samples': len(ambiguous_samples), 'empty_files': len(empty_file_ids),
          'incomplete_examples': incomplete[:100], 'ambiguous_examples': sorted(ambiguous_samples)[:100]}
report_path = INDEX_PATH.with_name('synthhuman_inventory_report_v3.json')
report_path.write_text(json.dumps(report, indent=2))
print(f"Complete samples: {len(complete_ids):,}; skipped incomplete: {len(incomplete):,}; ambiguous IDs: {len(ambiguous_samples):,}; empty files: {len(empty_file_ids):,}")
print('Inventory report:', report_path)
if len(complete_ids) < PILOT_SAMPLE_COUNT:
    raise ValueError(f'Only {len(complete_ids):,} complete samples exist in this inventory, below the requested {PILOT_SAMPLE_COUNT:,}. Check SOURCE_FOLDER_ID or set PILOT_SAMPLE_COUNT to an available count; missing labels cannot be fabricated.')
selected_ids = sorted(random.Random(SEED).sample(complete_ids, PILOT_SAMPLE_COUNT))
PILOT_FILE_IDS = {}
manifest_rows = []
for number in selected_ids:
    kinds = groups[number]
    for name in kinds.values():
        PILOT_FILE_IDS[name] = all_files[name]
    manifest_rows.append({'image': str(SYNTHHUMAN_ROOT / kinds['rgb']),
                          'depth': str(SYNTHHUMAN_ROOT / kinds['depth']),
                          'normal': str(SYNTHHUMAN_ROOT / kinds['normal']),
                          'mask': str(SYNTHHUMAN_ROOT / kinds['alpha']),
                          'subject_id': f'sample_{number:07d}', 'label_source': 'SynthHuman/DAViD renderer',
                          'depth_units': 'm', 'depth_scale_to_m': 0.01, 'normal_convention': 'camera_xyz_signed'})
# Separate the new sample selection from any previous pilot checkpoints/manifests.
MANIFEST = DATA_ROOT / 'pilot_manifests' / f'synthhuman_available_{PILOT_SAMPLE_COUNT}_seed{SEED}_v3.jsonl'
OUTPUT_DIR = DRIVE_ROOT / 'runs' / f'fastvit_t8_fpn_depth_normals_available{PILOT_SAMPLE_COUNT}_seed{SEED}_driveapi_v3'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
temporary = MANIFEST.with_suffix('.jsonl.tmp')
temporary.write_text(''.join(json.dumps(row) + '\n' for row in manifest_rows))
temporary.replace(MANIFEST)
print(f'Selected {len(manifest_rows):,} complete samples using seed {SEED}; missing groups were replaced from available data.')
print('Manifest:', MANIFEST)
print('New run directory:', OUTPUT_DIR)
# Keep only selected file metadata in RAM after the full inventory has been persisted.
del all_files, groups, state
if 'saved' in globals():
    del saved

In [ ]:
# 4. Read and validate the automatically generated manifest
def load_manifest(path,verify_paths=False):
    if not path.is_file(): raise FileNotFoundError(f'Manifest missing: {path}. Run the dataset manifest cell first.')
    rows=[json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    required={'image','depth','normal','mask','subject_id','label_source','depth_units','normal_convention'}
    if not rows: raise ValueError('Manifest contains no rows')
    for i,row in enumerate(rows):
        missing=required-row.keys()
        if missing: raise ValueError(f'row {i} missing fields: {sorted(missing)}')
        for key in ('image','depth','normal','mask'):
            p=Path(row[key]); p=p if p.is_absolute() else DATA_ROOT/p
            if verify_paths and not p.is_file(): raise FileNotFoundError(f'row {i} {key}: {p}')
            row[key]=str(p)
    return rows
ALL_ROWS=load_manifest(MANIFEST,verify_paths=False)
if DATASET_MODE == 'full':
    rows=ALL_ROWS
    RUN_MANIFEST=MANIFEST
elif DATASET_MODE == 'pilot':
    if len(ALL_ROWS)<PILOT_SAMPLE_COUNT:
        raise ValueError(f'Pilot requires {PILOT_SAMPLE_COUNT} complete labeled samples, but manifest has {len(ALL_ROWS)}.')
    pilot_rng=random.Random(SEED)
    rows=sorted(pilot_rng.sample(ALL_ROWS,PILOT_SAMPLE_COUNT),key=lambda r:r['subject_id'])
    RUN_MANIFEST=DATA_ROOT/'pilot_manifests'/f'depth_pilot_{PILOT_SAMPLE_COUNT}_seed{SEED}.jsonl'
    RUN_MANIFEST.parent.mkdir(parents=True,exist_ok=True)
    pilot_tmp=RUN_MANIFEST.with_suffix('.jsonl.tmp'); pilot_tmp.write_text(''.join(json.dumps(r)+'\n' for r in rows)); pilot_tmp.replace(RUN_MANIFEST)
else:
    raise ValueError("DATASET_MODE must be 'full' or 'pilot'")
print(f'Run mode={DATASET_MODE}; samples={len(rows):,}; active labels: depth + normals + mask')
print('Run manifest:',RUN_MANIFEST)

In [ ]:
# 5. Split by sample identity (SynthHuman release has no subject-ID field)
def split_by_subject(rows,seed=SEED):
    subjects=sorted({r['subject_id'] for r in rows})
    if len(subjects)<5:
        parts={'train':rows,'val':[],'test':[]}
        print(f'AUDIT MODE: only {len(subjects)} sample(s). This permits the loader audit only; training is blocked until a chunk is downloaded and at least five sample IDs are available.')
        return parts
    rng=random.Random(seed); rng.shuffle(subjects)
    n_test=max(1,round(len(subjects)*TEST_FRACTION)); n_val=max(1,round(len(subjects)*VAL_FRACTION))
    if n_test+n_val>=len(subjects): raise ValueError('Not enough samples for requested split fractions')
    test_ids=set(subjects[:n_test]); val_ids=set(subjects[n_test:n_test+n_val]); train_ids=set(subjects)-test_ids-val_ids
    parts={name:[r for r in rows if r['subject_id'] in ids] for name,ids in [('train',train_ids),('val',val_ids),('test',test_ids)]}
    split_info={'seed':seed,'split_unit':'sample_id (not verified person identity)','train_subjects':sorted(train_ids),'val_subjects':sorted(val_ids),'test_subjects':sorted(test_ids)}
    (OUTPUT_DIR/'split.json').write_text(json.dumps(split_info,indent=2))
    return parts
parts=split_by_subject(rows)
print({k:len(v) for k,v in parts.items()})
print('Evaluation limitation: split is sample-disjoint; the source release does not expose subject IDs for identity-disjoint evaluation.')

### Decode and validate source samples

These functions read only the local temporary files retrieved during preparation. No augmentation is baked into the shards.

In [ ]:
# 6. Read RGB/EXR labels and create a geometry-first dataset
os.environ['OPENCV_IO_ENABLE_OPENEXR']='1'
import cv2

def resize_array(arr,size,is_mask=False):
    arr=np.asarray(arr)
    if arr.ndim==2: arr=arr[...,None]
    mode=Image.Resampling.NEAREST if is_mask or arr.shape[-1]==1 else Image.Resampling.BILINEAR
    planes=[np.asarray(Image.fromarray(arr[...,c].astype(np.float32)).resize((size[1],size[0]),mode),dtype=np.float32) for c in range(arr.shape[-1])]
    return np.stack(planes,axis=-1)

# Paste into a new Colab cell after preprocessing and before shard preparation.
# Install the decoder first in another cell: %pip install "OpenEXR>=3.3,<4"
import os
from pathlib import Path
import numpy as np
from PIL import Image
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'


def read_exr_array(path, kind=None):
    path = Path(path)
    with path.open('rb') as handle:
        prefix = handle.read(16)
    if prefix[:4] != b'\x76\x2f\x31\x01':
        raise ValueError(f'{path.name} does not contain an OpenEXR header: first16_hex={prefix.hex()}, bytes={path.stat().st_size}. This is a source-file format/content problem, not a resize problem.')
    try:
        import OpenEXR
    except ImportError:
        import cv2
        try:
            array = cv2.imread(str(path), cv2.IMREAD_ANYCOLOR | cv2.IMREAD_ANYDEPTH)
        except cv2.error as error:
            raise RuntimeError('OpenCV cannot decode this EXR. Install OpenEXR with: %pip install "OpenEXR>=3.3,<4"') from error
        if array is None:
            raise RuntimeError(f'OpenCV returned None for {path.name}. Install OpenEXR with: %pip install "OpenEXR>=3.3,<4"')
        array = array[..., ::-1] if array.ndim == 3 else array
        if kind == 'depth' and array.ndim == 3:
            array = array[..., :1]
        return array.astype(np.float32)
    if not hasattr(OpenEXR, 'File'):
        raise RuntimeError('This reader requires OpenEXR >=3.3. Update the package before retrying.')
    try:
        with OpenEXR.File(str(path), separate_channels=True) as image:
            if len(image.parts) != 1:
                raise ValueError(f'{path.name}: multipart EXR is not supported for this label contract.')
            channels = image.channels()
            lookup = {name.upper(): name for name in channels}
            if kind == 'normal':
                if {'R', 'G', 'B'} <= lookup.keys():
                    names = [lookup[x] for x in ('R', 'G', 'B')]
                elif {'X', 'Y', 'Z'} <= lookup.keys():
                    names = [lookup[x] for x in ('X', 'Y', 'Z')]
                else:
                    raise ValueError(f'{path.name}: normals require R/G/B or X/Y/Z channels; found {sorted(channels)}.')
            elif kind == 'depth':
                if 'Z' in lookup:
                    names = [lookup['Z']]
                elif len(channels) == 1:
                    names = [next(iter(channels))]
                elif 'R' in lookup:
                    names = [lookup['R']]  # Preserves the prior RGB-depth red-channel convention.
                else:
                    raise ValueError(f'{path.name}: ambiguous depth channels: {sorted(channels)}.')
            else:
                raise ValueError('Declare EXR kind as depth or normal.')
            planes = []
            for name in names:
                pixels = channels[name].pixels
                if pixels.ndim != 2 or pixels.dtype.kind not in 'fiu':
                    raise ValueError(f'{path.name}: channel {name} is not a flat numeric image.')
                planes.append(np.array(pixels, dtype=np.float32, copy=True))
            return np.stack(planes, axis=-1)
    except Exception as error:
        raise ValueError(f'OpenEXR decode failed for {path.name}: {error}') from error


def read_array(path, rgb=False, mask=False, exr_kind=None):
    # Preparation passes local temporary paths, so no Drive FUSE access occurs here.
    path = Path(path)
    if path.suffix.lower() == '.exr':
        arr = read_exr_array(path, kind=exr_kind)
    elif path.suffix.lower() == '.npy':
        arr = np.load(path, allow_pickle=False)
    else:
        with Image.open(path) as image:
            arr = np.asarray(image.convert('L' if mask else 'RGB'))
    if rgb:
        if arr.dtype != np.uint8:
            arr = np.clip(arr*255 if arr.max() <= 1 else arr, 0, 255).astype(np.uint8)
        arr = np.asarray(Image.fromarray(arr).convert('RGB').resize((INPUT_SIZE[1], INPUT_SIZE[0]), Image.Resampling.BILINEAR), dtype=np.float32)/255.0
    elif mask:
        arr = resize_array((arr.squeeze() > 0).astype(np.uint8)*255, INPUT_SIZE, True)[..., 0] > 0
    else:
        arr = resize_array(arr.astype(np.float32), INPUT_SIZE).astype(np.float32)
    return arr

print('EXR decoder patch active: named channels, header check, OpenEXR reader with OpenCV fallback.')


class RelightingDataset(Dataset):
    def __init__(self,rows,augment=False): self.rows,self.augment=rows,augment
    def __len__(self): return len(self.rows)
    def __getitem__(self,idx):
        r=self.rows[idx]
        image=read_array(r['image'],rgb=True)
        depth=read_array(r['depth'],exr_kind='depth')
        normal=read_array(r['normal'],exr_kind='normal')
        mask=read_array(r['mask'],mask=True)
        if depth.shape[-1]!=1 or normal.shape[-1]!=3: raise ValueError(f'Bad channels at row {idx}: depth={depth.shape}, normal={normal.shape}')
        raw_depth=depth[...,0].copy()
        # SynthHuman uses 65504 as an invalid EXR depth sentinel (documented by DAViD's visualizer).
        # Filter in source units (cm) before converting to metres; otherwise 655.04 m leaks into training.
        valid_depth=np.isfinite(raw_depth)&(raw_depth>0)&(raw_depth<1000)&(~np.isclose(raw_depth,65504.0,rtol=0.0,atol=1.0))
        depth*=float(r.get('depth_scale_to_m',1.0))
        mask=mask&valid_depth&np.isfinite(normal).all(axis=-1)
        depth=np.nan_to_num(depth,nan=0.,posinf=0.,neginf=0.)
        normal=np.nan_to_num(normal,nan=0.,posinf=0.,neginf=0.)
        if self.augment and random.random()<.5:
            image,depth,normal,mask=[np.flip(a,axis=1).copy() for a in (image,depth,normal,mask)]
            normal[...,0]*=-1
        image=(image-np.array([.485,.456,.406],np.float32))/np.array([.229,.224,.225],np.float32)
        tensors=[torch.from_numpy(a).permute(2,0,1).float() for a in (image,depth,normal)]
        tensors.append(torch.from_numpy(mask[None].astype(np.float32)))
        tensors[2]=F.normalize(tensors[2],dim=0,eps=1e-6)
        tensors[3]*=(torch.linalg.vector_norm(tensors[2],dim=0,keepdim=True)>0).float()
        return tuple(tensors)

### Pack two shards first, then continue the full selection

Set RUN_PREPARATION=True below. Keep PREPARE_SHARD_LIMIT=2 for the 500-sample benchmark; set it to None and rerun after reviewing time and size. Start with FETCH_WORKERS=8; reduce to 4 if repeated Drive rate-limit retries occur.

In [ ]:
# Embedded in prepare_synthhuman_shards.ipynb after its local-path preprocessing cell.
# Use that notebook; the older direct-API training reader is not the preparation decoder.
import concurrent.futures
import copy
import hashlib
import io
import shutil
import tarfile
import tempfile
import threading
from tqdm.auto import tqdm

SHARD_SAMPLES = 250
FETCH_WORKERS = 8
SHARD_SCRATCH_ROOT = Path('/content')
PREPARE_SHARD_LIMIT = 2  # 2 shards = 500 preparation samples; use None for all 50k.
RUN_PREPARATION = False  # Set True when ready for reads from your existing Drive.
SHARD_ROOT = DATA_ROOT / 'prepared_shards' / f'synthhuman_{PILOT_SAMPLE_COUNT}_seed{SEED}_{INPUT_SIZE[0]}x{INPUT_SIZE[1]}_v1'
SHARD_MANIFEST_PATH = SHARD_ROOT / 'manifest.json'


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()


def prepare_existing_drive_shards():
    active_reader = globals().get('read_array')
    if active_reader is not None and 'read_exr_array' not in active_reader.__code__.co_names:
        raise RuntimeError('The old OpenCV-only read_array is still active. Run activate_exr_decoder.py AFTER preprocessing, then rerun ONLY preparation.')
    if SHARD_SAMPLES < 1 or FETCH_WORKERS < 1:
        raise ValueError('SHARD_SAMPLES and FETCH_WORKERS must be positive.')
    if PREPARE_SHARD_LIMIT is not None and PREPARE_SHARD_LIMIT < 1:
        raise ValueError('PREPARE_SHARD_LIMIT must be positive or None.')
    SHARD_ROOT.mkdir(parents=True, exist_ok=True)
    selection = [(part, row['subject_id']) for part in ('train', 'val', 'test') for row in parts[part]]
    config = {'format_version': 1, 'samples': len(selection), 'seed': SEED,
              'input_size': list(INPUT_SIZE), 'source_folder_id': SOURCE_FOLDER_ID,
              'selection_sha256': hashlib.sha256(json.dumps(selection).encode()).hexdigest(),
              'shard_samples': SHARD_SAMPLES, 'depth_units': 'm',
              'depth_dtype': 'float32', 'normal_dtype': 'float32',
              'rgb_dtype': 'uint8', 'mask_dtype': 'uint8',
              'preprocessing': 'geometry_dataset_v1_no_augmentation'}
    packed = {'config': config, 'shards': [], 'complete': False}
    if SHARD_MANIFEST_PATH.is_file():
        packed = json.loads(SHARD_MANIFEST_PATH.read_text())
        if packed['config'] != config:
            raise ValueError('Prepared dataset configuration differs. Use a new SHARD_ROOT instead of mixing datasets.')
    existing = {item['name']: item for item in packed['shards']}
    plan = []
    for part in ('train', 'val', 'test'):
        for offset in range(0, len(parts[part]), SHARD_SAMPLES):
            name = f'{part}_{offset // SHARD_SAMPLES:05d}.tar'
            plan.append((part, name, parts[part][offset:offset+SHARD_SAMPLES]))
    chosen = plan if PREPARE_SHARD_LIMIT is None else plan[:PREPARE_SHARD_LIMIT]
    todo = []
    for part, name, batch_rows in chosen:
        item = existing.get(name)
        path = SHARD_ROOT / name
        if item and path.is_file() and path.stat().st_size == item['bytes']:
            if [r['subject_id'] for r in item['rows']] != [r['subject_id'] for r in batch_rows]:
                raise ValueError(f'Sample membership changed for {name}.')
        else:
            todo.append((part, name, batch_rows))

    def save_manifest():
        packed['shards'] = [existing[name] for _, name, _ in plan if name in existing]
        packed['complete'] = len(packed['shards']) == len(plan)
        temporary = SHARD_MANIFEST_PATH.with_suffix('.json.part')
        temporary.write_text(json.dumps(packed))
        temporary.replace(SHARD_MANIFEST_PATH)

    save_manifest()
    print(f'Planned dataset: {len(selection):,} samples / {len(plan)} shards. This run: {sum(len(x[2]) for x in todo):,} new samples.')
    client_local = threading.local()

    def fetch(path, destination):
        from googleapiclient.discovery import build
        import google_auth_httplib2, httplib2
        if not hasattr(client_local, 'client'):
            client_local.client = build('drive', 'v3', http=google_auth_httplib2.AuthorizedHttp(
                copy.copy(credentials), http=httplib2.Http(timeout=120)), cache_discovery=False)
        name = Path(path).name
        metadata = PILOT_FILE_IDS[name]
        content = client_local.client.files().get_media(fileId=metadata['id']).execute(num_retries=5)
        if len(content) != metadata['size']:
            raise IOError(f'Incomplete API read: {name}')
        destination.write_bytes(content)
        return len(content)

    started = time.perf_counter()
    done_samples = 0
    uploaded_bytes = 0
    progress = tqdm(total=sum(len(x[2]) for x in todo), desc='Prepare existing dataset', unit='sample')
    try:
        with tempfile.TemporaryDirectory(dir=SHARD_SCRATCH_ROOT, prefix='synthhuman_pack_') as scratch, concurrent.futures.ThreadPoolExecutor(max_workers=FETCH_WORKERS) as pool:
            scratch = Path(scratch)
            for part, name, shard_rows in todo:
                shard_start = time.perf_counter()
                archive_path = scratch / name
                prepared_rows = []
                with tarfile.open(archive_path, 'w') as archive:
                    for start in range(0, len(shard_rows), 8):
                        source_rows = shard_rows[start:start+8]
                        expected_bytes = sum(PILOT_FILE_IDS[Path(r[k]).name]['size'] for r in source_rows for k in ('image', 'depth', 'normal', 'mask'))
                        if shutil.disk_usage(scratch).free < expected_bytes + 1024**3:
                            raise OSError('Insufficient local scratch space. Completed shards are retained on Drive.')
                        with tempfile.TemporaryDirectory(dir=scratch, prefix='raw_') as raw:
                            raw = Path(raw)
                            local_rows = []
                            futures = []
                            for row in source_rows:
                                local = dict(row)
                                for key in ('image', 'depth', 'normal', 'mask'):
                                    target = raw / Path(row[key]).name
                                    futures.append(pool.submit(fetch, row[key], target))
                                    local[key] = str(target)
                                local_rows.append(local)
                            try:
                                for future in concurrent.futures.as_completed(futures):
                                    future.result()
                            finally:
                                for future in futures:
                                    future.cancel()
                                concurrent.futures.wait(futures)
                            dataset = RelightingDataset(local_rows, augment=False)
                            for index, row in enumerate(source_rows):
                                image, depth, normal, mask = dataset[index]
                                if not all(torch.isfinite(t).all() for t in (image, depth, normal, mask)) or mask.sum() <= 0:
                                    raise ValueError(f'Invalid decoded sample: {row["subject_id"]}. No placeholder label was written.')
                                # Store resized RGB without ImageNet normalization; geometric maps keep float32 precision.
                                rgb = ((image.numpy() * np.array([.229,.224,.225], np.float32)[:,None,None] + np.array([.485,.456,.406], np.float32)[:,None,None]) * 255).round().clip(0,255).astype(np.uint8)
                                member = row['subject_id'] + '.npz'
                                payload = io.BytesIO()
                                np.savez_compressed(payload, rgb=rgb, depth=depth.numpy().astype(np.float32), normal=normal.numpy().astype(np.float32), mask=mask.numpy().astype(np.uint8))
                                data = payload.getvalue()
                                info = tarfile.TarInfo(member)
                                info.size = len(data)
                                info.mtime = 0
                                archive.addfile(info, io.BytesIO(data))
                                prepared_rows.append({**row, 'partition': part, 'shard': name, 'member': member})
                                progress.update(1)
                                done_samples += 1
                                progress.set_postfix(shard=name, samples_s=f'{done_samples/max(time.perf_counter()-started,1e-9):.2f}')
                checksum = file_sha256(archive_path)
                size = archive_path.stat().st_size
                remote_temporary = SHARD_ROOT / (name + '.part')
                shutil.copyfile(archive_path, remote_temporary)
                if remote_temporary.stat().st_size != size:
                    raise IOError(f'Incomplete shard upload: {name}')
                remote_temporary.replace(SHARD_ROOT / name)
                existing[name] = {'name': name, 'partition': part, 'bytes': size, 'sha256': checksum, 'rows': prepared_rows}
                save_manifest()
                archive_path.unlink()
                uploaded_bytes += size
                print(f'{name}: {len(prepared_rows)} samples saved to Drive; {size/1024**2:.1f} MiB; {time.perf_counter()-shard_start:.1f}s')
    finally:
        progress.close()
    elapsed = time.perf_counter() - started
    if done_samples:
        print(f'Preparation measurement: {done_samples} samples in {elapsed/60:.1f} minutes; {uploaded_bytes/1024**2/done_samples:.2f} MiB/sample packed.')
        print(f'Extrapolation for {len(selection):,} samples: {elapsed/done_samples*len(selection)/3600:.1f} hours preparation; distribution/network can change this.')
    print('Ready for full training:', packed['complete'])
    print('Manifest:', SHARD_MANIFEST_PATH)
    if not packed['complete']:
        print('To continue all 50k samples, set PREPARE_SHARD_LIMIT=None and rerun this cell. Finished shards are skipped.')

if RUN_PREPARATION:
    prepare_existing_drive_shards()
else:
    print('Preparation disabled. Set RUN_PREPARATION=True for the first 500 samples; PREPARE_SHARD_LIMIT=None continues the full selection.')


In [ ]:
if SHARD_MANIFEST_PATH.is_file():
    summary=json.loads(SHARD_MANIFEST_PATH.read_text())
    print('Finished shards:',len(summary['shards']))
    print('Packed samples:',sum(len(s['rows']) for s in summary['shards']))
    print('Full selection ready:',summary['complete'])
    print('Training notebook: training_pipeline_relighting.ipynb; select a GPU after full preparation finishes.')
else:
    print('No shards prepared yet. Enable RUN_PREPARATION in the previous cell.')